# 02 Ground-Truth Annotation

## Version 1

Click restart session before running the next code.

In [ ]:
!pip install mediapipe==0.10.13
!wget -q https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task -O face_landmarker.task

In [ ]:
import cv2
import os
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

INPUT_DIR = "/content/drive/Shareddrives/U-Net Ensemble/Dataset/Normalized/Synthetic/norm/template-c"
OUTPUT_MASK_DIR = "/content/drive/Shareddrives/U-Net Ensemble/Dataset/Masks/Synthetic/temp"
OUTPUT_OVERLAY_DIR = "/content/drive/Shareddrives/U-Net Ensemble/Dataset/Annotated/Synthetic/temp"

os.makedirs(OUTPUT_OVERLAY_DIR, exist_ok=True)
os.makedirs(OUTPUT_MASK_DIR, exist_ok=True)

FACE_OVAL_INDICES = [
    10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361, 288,
    397, 365, 379, 378, 400, 377, 152, 148, 176, 149, 150, 136,
    172, 58, 132, 93, 234, 127, 162, 21, 54, 103, 67, 109
]

base_options = python.BaseOptions(model_asset_path="face_landmarker.task")
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

with vision.FaceLandmarker.create_from_options(options) as landmarker:
    for filename in os.listdir(INPUT_DIR):
        if filename.lower().endswith((".jpg", ".png", ".jpeg")):
            img_path = os.path.join(INPUT_DIR, filename)
            image_bgr = cv2.imread(img_path)

            if image_bgr is None:
                print(f"Could not read image: {filename}, skipping.")
                continue

            ih, iw = image_bgr.shape[:2]
            image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)

            results = landmarker.detect(mp_image)
            mask = np.zeros((ih, iw), dtype=np.uint8)
            annotated = image_bgr.copy()

            if results.face_landmarks:
                for face_landmarks in results.face_landmarks:
                    # Extract face oval polygon points
                    oval_points = []
                    for idx in FACE_OVAL_INDICES:
                        lm = face_landmarks[idx]
                        x = int(lm.x * iw)
                        y = int(lm.y * ih)
                        oval_points.append([x, y])

                    oval_points = np.array(oval_points, dtype=np.int32)

                    # Fill binary mask
                    cv2.fillPoly(mask, [oval_points], 255)

                # --- FIXED BLENDING ---
                # Convert mask to float32 for safe math
                mask_float = mask.astype(np.float32) / 255.0
                mask_3ch = np.stack([mask_float, mask_float, mask_float], axis=2)

                # Work in float32
                base = annotated.astype(np.float32)

                # Build green channel boost within the mask
                green_tint = np.zeros_like(base)
                green_tint[:, :, 1] = 255.0  # green channel only (BGR: index 1)

                alpha = 0.3
                blended = base * (1.0 - alpha * mask_3ch) + green_tint * (alpha * mask_3ch)
                annotated = np.clip(blended, 0, 255).astype(np.uint8)

            cv2.imwrite(os.path.join(OUTPUT_OVERLAY_DIR, filename), annotated)
            cv2.imwrite(os.path.join(OUTPUT_MASK_DIR, filename), mask)
            print(f"Processed: {filename}")

print("Annotation and mask generation complete!")

## Version 2

Click restart session before running the next code.

In [ ]:
!pip install mediapipe==0.10.13
!wget -q https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task -O face_landmarker.task
!wget -q https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite -O face_detector.tflite

In [ ]:
import cv2
import os
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

INPUT_DIR = "/content/drive/Shareddrives/U-Net Ensemble/Dataset/Normalized/Synthetic/temp"
OUTPUT_MASK_DIR = "/content/drive/Shareddrives/U-Net Ensemble/Dataset/Masks/Synthetic-1/norm"
OUTPUT_OVERLAY_DIR = "/content/drive/Shareddrives/U-Net Ensemble/Dataset/Annotated/Synthetic-1/norm"

os.makedirs(OUTPUT_OVERLAY_DIR, exist_ok=True)
os.makedirs(OUTPUT_MASK_DIR, exist_ok=True)

FACE_OVAL_INDICES = [
    10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361, 288,
    397, 365, 379, 378, 400, 377, 152, 148, 176, 149, 150, 136,
    172, 58, 132, 93, 234, 127, 162, 21, 54, 103, 67, 109
]

# Landmarker with lowered thresholds
lm_base = python.BaseOptions(model_asset_path="face_landmarker.task")
lm_options = vision.FaceLandmarkerOptions(
    base_options=lm_base,
    num_faces=1,
    min_face_detection_confidence=0.3,  # lowered from 0.5
    min_face_presence_confidence=0.3,   # lowered from 0.5
    min_tracking_confidence=0.3         # lowered from 0.5
)

# Fallback face detector
fd_base = python.BaseOptions(model_asset_path="face_detector.tflite")
fd_options = vision.FaceDetectorOptions(
    base_options=fd_base,
    min_detection_confidence=0.2        # lowered for difficult images
)

def apply_green_tint(image_bgr, mask, alpha=0.45):
    mask_float = mask.astype(np.float32) / 255.0
    mask_3ch = np.stack([mask_float, mask_float, mask_float], axis=2)
    base = image_bgr.astype(np.float32)
    green_tint = np.zeros_like(base)
    green_tint[:, :, 1] = 255.0
    blended = base * (1.0 - alpha * mask_3ch) + green_tint * (alpha * mask_3ch)
    return np.clip(blended, 0, 255).astype(np.uint8)

with vision.FaceLandmarker.create_from_options(lm_options) as landmarker, \
     vision.FaceDetector.create_from_options(fd_options) as face_detector:

    for filename in os.listdir(INPUT_DIR):
        if filename.lower().endswith((".jpg", ".png", ".jpeg")):
            img_path = os.path.join(INPUT_DIR, filename)
            image_bgr = cv2.imread(img_path)

            if image_bgr is None:
                print(f"Could not read image: {filename}, skipping.")
                continue

            ih, iw = image_bgr.shape[:2]
            image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)

            mask = np.zeros((ih, iw), dtype=np.uint8)
            annotated = image_bgr.copy()
            detection_method = None

            # --- Try Face Landmarker first (tight face polygon) ---
            lm_results = landmarker.detect(mp_image)
            if lm_results.face_landmarks:
                for face_landmarks in lm_results.face_landmarks:
                    oval_points = []
                    for idx in FACE_OVAL_INDICES:
                        lm = face_landmarks[idx]
                        x = int(lm.x * iw)
                        y = int(lm.y * ih)
                        oval_points.append([x, y])
                    oval_points = np.array(oval_points, dtype=np.int32)
                    cv2.fillPoly(mask, [oval_points], 255)
                detection_method = "landmarker"

            # --- Fallback: use bounding box from face detector ---
            else:
                fd_results = face_detector.detect(mp_image)
                if fd_results.detections:
                    for detection in fd_results.detections:
                        bbox = detection.bounding_box
                        x = max(0, bbox.origin_x)
                        y = max(0, bbox.origin_y)
                        x2 = min(iw, bbox.origin_x + bbox.width)
                        y2 = min(ih, bbox.origin_y + bbox.height)
                        # Draw ellipse to approximate face shape instead of rectangle
                        cx, cy = (x + x2) // 2, (y + y2) // 2
                        rx, ry = (x2 - x) // 2, (y2 - y) // 2
                        cv2.ellipse(mask, (cx, cy), (rx, ry), 0, 0, 360, 255, -1)
                    detection_method = "fallback_detector"

            if detection_method:
                annotated = apply_green_tint(image_bgr, mask)
                print(f"Processed [{detection_method}]: {filename}")
            else:
                print(f"WARNING - No face detected: {filename}")

            cv2.imwrite(os.path.join(OUTPUT_OVERLAY_DIR, filename), annotated)
            cv2.imwrite(os.path.join(OUTPUT_MASK_DIR, filename), mask)

print("Annotation and mask generation complete!")